[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Train YOLO11 Object Detection on a Custom Dataset

---

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/ultralytics/ultralytics)

YOLO11 builds on the advancements introduced in YOLOv9 and YOLOv10 earlier this year, incorporating improved architectural designs, enhanced feature extraction techniques, and optimized training methods.

YOLO11m achieves a higher mean mAP score on the COCO dataset while using 22% fewer parameters than YOLOv8m, making it computationally lighter without sacrificing performance.

YOLOv11 is available in 5 different sizes, ranging from `2.6M` to `56.9M` parameters, and capable of achieving from `39.5` to `54.7` mAP on the COCO dataset.

## Setup

### Configure API keys

To fine-tune YOLO11, you need to provide your Roboflow API key. Follow these steps:

- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

### Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Install YOLO11 via Ultralytics

In [ ]:
%pip install "ultralytics<=8.3.40" supervision roboflow
# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()

## Download Custom Dataset for Evaluation

**NOTE:** When training YOLOv11, make sure your data is located in `datasets`. If you'd like to change the default location of the data you want to use for fine-tuning, you can do so through Ultralytics' `settings.json`. In this tutorial, we will use one of the [datasets](https://universe.roboflow.com/liangdianzhong/-qvdww) available on [Roboflow Universe](https://universe.roboflow.com/). When downloading, make sure to select the `yolov11` export format.

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

workspace = rf.workspace("krina-shukal")
project = workspace.project("lab1-dataset")
version = project.version(2)
dataset = version.download("yolov11")

## Import Saved Fine-Tuned YOLO11 Model

In [ ]:
from google.colab import userdata
import requests
import os

github_token = userdata.get("GITHUB_TOKEN")
github_username = "krs524"

model_url = (
    f"https://api.github.com/repos/{github_username}/GenAI_Lab1/"
    "contents/Step2/models/best.pt"
)

response = requests.get(
    model_url,
    headers={
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github.raw+json"
    },
    timeout=120
)

response.raise_for_status()

saved_model_path = "/content/github_best.pt"

with open(saved_model_path, "wb") as file:
    file.write(response.content)

print("Saved model downloaded successfully.")
print("Model file size:", round(os.path.getsize(saved_model_path) / 1024**2, 2), "MiB")

from ultralytics import YOLO

saved_model = YOLO(saved_model_path)
print("Model classes:", saved_model.names)

## Validate fine-tuned model

In [ ]:
!yolo task=detect mode=val model=/content/github_best.pt data={dataset.location}/data.yaml imgsz=640

## Inference with Saved Fine-Tuned Model on D

In [ ]:
!yolo task=detect mode=predict model=/content/github_best.pt conf=0.25 source={dataset.location}/test/images save=True

**NOTE:** Let's take a look at few results.

In [ ]:
from google.colab import userdata
import requests

github_token = userdata.get("GITHUB_TOKEN")

github_username = "krs524"

image_url = (
    f"https://api.github.com/repos/{github_username}/GenAI_Lab1/"
    "contents/Step1/test_images/Hanger.jpg"
)

response = requests.get(
    image_url,
    headers={
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github.raw+json"
    },
    timeout=30
)

response.raise_for_status()

with open("/content/Hanger.jpg", "wb") as file:
    file.write(response.content)

print("Hanger.jpg downloaded successfully from GitHub.")

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/github_best.pt")

model.predict(
    source="/content/Hanger.jpg",
    conf=0.25,
    save=True
)